# MRI Segmentation Project: Left Ventricle Detection

Notebook centré sur la visualisation des données du projet.

---

## Table of Contents

1. [Setup](#setup)
2. [Data Exploration](#data-exploration)
   - [Dataset Structure](#dataset-structure)
   - [Image Shape Distribution](#image-shape-distribution)
3. [Visualizing Patient Data](#visualizing-patient-data)
   - [Viewing Slices and Frames](#viewing-slices-and-frames)
4. [3D Visualization](#3d-visualization)

---

## Setup <a name="setup"></a>

In [ ]:
# Importing required libraries
import os
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = 'notebook'  # Use 'colab' if using Google Colab

# Enable inline plots for matplotlib
%matplotlib inline

---

## Data Exploration <a name="data-exploration"></a>

### Dataset Structure <a name="dataset-structure"></a>

The dataset is organized as follows:

- `database/`
  - `training/`
    - `patient001/`
      - `info.cfg`
      - `patient001_4d.nii.gz`
      - `patient001_frame01.nii.gz`
      - `patient001_frame01_gt.nii.gz`
      - `patient001_frame12.nii.gz`
      - `patient001_frame12_gt.nii.gz`
    - `patient002/`
    - `...`
  - `testing/`
    - `patient101/`
    - `...`
    - `patient150/`

---

### Image Shape Distribution <a name="image-shape-distribution"></a>

In [ ]:
training_base_path = 'database/training/'
testing_base_path = 'database/testing/'

patient_folders = [folder for folder in os.listdir(training_base_path)]
print(f"Found {len(patient_folders)} patients in the training dataset.")

In [ ]:
image_shapes = []

for folder in patient_folders:
    image_path = os.path.join(training_base_path, folder, f'{folder}_4d.nii.gz')
    
    image = nib.load(image_path)
    data = image.get_fdata()
    
    image_shapes.append(data.shape)

# Convert the list to a NumPy array for easier manipulation
image_shapes = np.array(image_shapes)

In [ ]:
# Display the range of shapes
print("Image Shape Statistics:")
print(f"  Width: min = {image_shapes[:, 0].min()}, max = {image_shapes[:, 0].max()}")
print(f"  Height: min = {image_shapes[:, 1].min()}, max = {image_shapes[:, 1].max()}")
print(f"  Number of Slices: min = {image_shapes[:, 2].min()}, max = {image_shapes[:, 2].max()}")
print(f"  Time Frames: min = {image_shapes[:, 3].min()}, max = {image_shapes[:, 3].max()}")

In [ ]:
# 3D Scatter Plot of image shapes
fig = go.Figure(data=[
    go.Scatter3d(
        x=image_shapes[:, 0],
        y=image_shapes[:, 1],
        z=image_shapes[:, 2],
        mode='markers',
        marker=dict(size=5)
    )
])

fig.update_layout(
    title='Distribution of Image Shapes in Training Dataset',
    scene=dict(
        xaxis_title='Width',
        yaxis_title='Height',
        zaxis_title='Number of Slices'
    )
)

fig.show()

---

### Viewing Slices and Frames <a name="viewing-slices-and-frames"></a>

In [ ]:
# Specify the patient number to visualize
patient_number = 1
patient_id = f'patient{patient_number:03d}'

# Define paths to patient data
patient_path = os.path.join(training_base_path, patient_id)
image_4d_path = os.path.join(patient_path, f'{patient_id}_4d.nii.gz')
frame01_path = os.path.join(patient_path, f'{patient_id}_frame01.nii.gz')
frame01_gt_path = os.path.join(patient_path, f'{patient_id}_frame01_gt.nii.gz')

# Load images
image_4d = nib.load(image_4d_path).get_fdata()
frame01 = nib.load(frame01_path).get_fdata()
frame01_gt = nib.load(frame01_gt_path).get_fdata()

print(f"4D Image Shape: {image_4d.shape}")  # (Width, Height, Slices, Time Frames)
print(f"Frame01 Image Shape: {frame01.shape}")
print(f"Frame01 Ground Truth Shape: {frame01_gt.shape}")

In [ ]:
# Select a slice and time frame to visualize
slice_index = frame01.shape[2] // 2  # Middle slice

plt.imshow(frame01[:, :, slice_index], cmap='gray')
plt.title(f'Patient {patient_number} - Frame01 - Slice {slice_index}')
plt.axis('off')
plt.show()

In [ ]:
# Plot the ground truth segmentation for frame01
plt.imshow(frame01_gt[:, :, slice_index], cmap='gray')
plt.title(f'Patient {patient_number} - Frame01 Ground Truth - Slice {slice_index}')
plt.axis('off')
plt.show()

---

## 3D Visualization <a name="3d-visualization"></a>

Based on [Left ventricle using CNN](https://www.kaggle.com/code/abimbolasalawu/left-ventricle-using-cnn/notebook).


In [ ]:
time_frame = 0  # Adjust as needed

volume = image_4d[:, :, :, time_frame]
volume = volume.T  # Transpose to match Plotly's axis order
slices, height, width = volume.shape

# Create frames for animation
fig = go.Figure(frames=[
    go.Frame(
        data=go.Surface(
            z=k * np.ones((height, width)),
            surfacecolor=volume[slices - 1 - k],
            cmin=volume.min(),
            cmax=volume.max(),
            colorscale='Gray',
            showscale=False,
        ),
        name=str(k)
    )
    for k in range(slices)
])

# Add the initial surface
fig.add_trace(go.Surface(
    z=(slices - 1) * np.ones((height, width)),
    surfacecolor=volume[slices - 1],
    colorscale='Gray',
    cmin=volume.min(),
    cmax=volume.max(),
    colorbar=dict(thickness=20, ticklen=4)
))

# Define animation settings
def frame_args(duration):
    return {
        "frame": {"duration": duration},
        "mode": "immediate",
        "fromcurrent": True,
        "transition": {"duration": duration, "easing": "linear"},
    }

# Configure sliders and buttons
sliders = [{
    "pad": {"b": 10, "t": 60},
    "len": 0.9,
    "x": 0.1,
    "y": 0,
    "steps": [
        {
            "args": [[f.name], frame_args(0)],
            "label": str(k),
            "method": "animate",
        }
        for k, f in enumerate(fig.frames)
    ],
}]

fig.update_layout(
    title='3D Visualization of MRI Slices',
    width=800,
    height=800,
    scene=dict(
        zaxis=dict(range=[-1, slices], autorange=False),
        aspectratio=dict(x=1, y=1, z=1),
        xaxis_title='Width',
        yaxis_title='Height',
        zaxis_title='Slices',
    ),
    updatemenus=[
        {
            "buttons": [
                {
                    "args": [None, frame_args(50)],
                    "label": "&#9654;",  # Play symbol
                    "method": "animate",
                },
                {
                    "args": [[None], frame_args(0)],
                    "label": "&#9724;",  # Pause symbol
                    "method": "animate",
                },
            ],
            "direction": "left",
            "pad": {"r": 10, "t": 70},
            "type": "buttons",
            "x": 0.1,
            "y": 0,
        }
    ],
    sliders=sliders
)

fig.show()